<div align="center">

# 🛒 Customer Segmentation & Churn Analysis
### Online Retail II Dataset · RFM Feature Engineering · K-Means Clustering

---

[![Python](https://img.shields.io/badge/Python-3.10+-blue?logo=python)](https://www.python.org/)
[![Scikit-learn](https://img.shields.io/badge/Scikit--learn-1.x-orange?logo=scikit-learn)](https://scikit-learn.org/)
[![Pandas](https://img.shields.io/badge/Pandas-2.x-150458?logo=pandas)](https://pandas.pydata.org/)
[![License: MIT](https://img.shields.io/badge/License-MIT-green.svg)](https://opensource.org/licenses/MIT)

</div>


---

## 📌 Project Overview

This end-to-end Data Analysis project transforms raw transactional e-commerce data into **actionable business intelligence** using unsupervised machine learning.

**Business Problem:**  
A UK-based online retailer wants to understand its customer base better — who are the most valuable customers, who is at risk of churning, and how can marketing efforts be personalised?

**Solution:**  
We engineer RFM-based features at the customer level, develop a **behaviour-driven churn flag**, and apply **K-Means clustering** to segment customers into 8 distinct, business-interpretable groups.

---

## 🗂️ Table of Contents

| # | Section |
|---|---------|
| 1 | [Dataset Overview](#1-dataset-overview) |
| 2 | [Data Cleaning & Preprocessing](#2-data-cleaning--preprocessing) |
| 3 | [Feature Engineering](#3-feature-engineering) |
| 4 | [Churn Identification](#4-churn-identification) |
| 5 | [Exploratory Data Analysis (EDA)](#5-exploratory-data-analysis-eda) |
| 6 | [Customer Segmentation via K-Means](#6-customer-segmentation-via-k-means) |
| 7 | [Segment Profiles & Business Insights](#7-segment-profiles--business-insights) |
| 8 | [PCA Visualisation](#8-pca-visualisation) |
| 9 | [Key Takeaways & Recommendations](#9-key-takeaways--recommendations) |

---

## 📦 Dataset

| Attribute | Details |
|-----------|---------|
| **Source** | [UCI Machine Learning Repository — Online Retail II](https://archive.ics.uci.edu/ml/datasets/Online+Retail+II) |
| **Period** | 01 Dec 2009 – 09 Dec 2011 |
| **Records** | ~1 Million transactions |
| **Scope** | UK-based non-store online retailer |

**Key Columns**

| Column | Description |
|--------|-------------|
| `Invoice` | Unique invoice number; prefix `C` = cancellation |
| `StockCode` | Product code |
| `Description` | Product name |
| `Quantity` | Units purchased (negative = return) |
| `InvoiceDate` | Date & time of invoice |
| `Price` | Unit price in GBP |
| `Customer ID` | Unique customer identifier |
| `Country` | Customer country |


---

## 1. Dataset Overview

> **Note:** All library imports are consolidated here for clarity and reproducibility.

In [ ]:
# ── Core Libraries ────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ── Machine Learning ──────────────────────────────────────────────────────────
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

# ── Plot Styling ──────────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
})

print("✅ All libraries loaded successfully.")


In [ ]:
# ── Load Dataset ──────────────────────────────────────────────────────────────
df = pd.read_excel("online_retail_II.xlsx")
clean_df = df.copy()   # preserve original snapshot

print(f"Shape  : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Memory : {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
df.sample(5)


In [ ]:
# ── Data Types & Missing Values ───────────────────────────────────────────────
print("=" * 50)
print("COLUMN INFO")
print("=" * 50)
df.info()

print("\n" + "=" * 50)
print("MISSING VALUES (%)")
print("=" * 50)
missing = (df.isnull().mean() * 100).round(2)
print(missing[missing > 0].to_string())


In [ ]:
# ── Unique Cardinality ────────────────────────────────────────────────────────
print("Unique values per column:")
df.nunique().to_frame("Unique Count")


---

## 2. Data Cleaning & Preprocessing

### 🔍 Issues Identified

| Issue | Observation | Action Taken |
|-------|-------------|--------------|
| Missing `Customer ID` | ~24.9% of rows | Dropped — anonymous transactions not useful for customer-level analysis |
| Negative `Quantity` | Returns / cancellations | Identified; retained for awareness |
| Negative `Price` | Outlet/adjustment records | Filtered out via `Description` keyword match |
| Cancelled Invoices (`C` prefix) | Credit notes | Identified; not separately removed (captured by negative quantity) |
| Duplicate rows | Exact row duplicates | Dropped |


In [ ]:
# ── Cancelled Invoices Overview ───────────────────────────────────────────────
cancelled = df[df["Invoice"].astype(str).str.startswith("C")]
print(f"Cancelled invoices : {cancelled['Invoice'].nunique():,} unique invoices")
print(f"Rows affected      : {len(cancelled):,}")


In [ ]:
# ── Returned Products ─────────────────────────────────────────────────────────
pct_returns = (df["Quantity"] < 0).mean() * 100
print(f"Rows with negative Quantity (returns) : {pct_returns:.2f}%")


In [ ]:
# ── Remove Duplicates ─────────────────────────────────────────────────────────
before = len(df)
df.drop_duplicates(inplace=True)
print(f"Duplicate rows removed : {before - len(df):,}")
print(f"Remaining rows         : {len(df):,}")


In [ ]:
# ── Drop Rows Without Customer ID ─────────────────────────────────────────────
before = len(df)
df.dropna(subset=["Customer ID"], inplace=True)
print(f"Rows dropped (no Customer ID) : {before - len(df):,}")
print(f"Remaining rows                : {len(df):,}")
print(f"Unique customers remaining    : {df['Customer ID'].nunique():,}")


In [ ]:
# ── Remove Accounting / Adjustment Entries ────────────────────────────────────
before = len(df)
df = df[~df["Description"].astype(str).str.contains("Adjust", case=False)]
print(f"Adjustment rows removed : {before - len(df):,}")
print(f"Final clean dataset     : {len(df):,} rows")


In [ ]:
# ── Derive Revenue Column ─────────────────────────────────────────────────────
df["Total"] = df["Price"] * df["Quantity"]
print("✅ 'Total' (revenue per line item) column created.")
df[["Price", "Quantity", "Total"]].describe().round(2)


---

## 3. Feature Engineering

We transform the transaction-level dataset into a **customer-level feature table** using RFM-inspired metrics plus behavioural features.

| Feature | Description | Engineering Logic |
|---------|-------------|-------------------|
| `Recency` | Days since last purchase | `max(InvoiceDate)` across dataset − customer's last purchase date |
| `Total Invoice` (Frequency) | Number of distinct orders | `nunique(Invoice)` per customer |
| `Monetary` | Total revenue from customer | `sum(Total)` per customer |
| `Avg_expenses_per_invoice` | Average order value | `Monetary / Total Invoice` |
| `customer_span` | Days from first to last purchase | `max(InvoiceDate) − min(InvoiceDate)` per customer |
| `gap_days` | Average days between consecutive purchases | Derived from invoice-level sorted dates |

> **`gap_days` is the most novel feature** — it captures a customer's natural buying rhythm rather than just counting purchases.


In [ ]:
# ── Build Customer-Level Feature Table ────────────────────────────────────────
tf = df.groupby("Customer ID")["InvoiceDate"].max().reset_index()

# Recency: days since last purchase (relative to dataset end date)
reference_date = df["InvoiceDate"].max()
tf["Recency"] = (reference_date - tf["InvoiceDate"]).dt.days

# Frequency: unique invoices
tf = tf.merge(
    df.groupby("Customer ID")["Invoice"].nunique().reset_index().rename(columns={"Invoice": "Total Invoice"}),
    on="Customer ID", how="left"
)

# Monetary: total spend
tf = tf.merge(
    df.groupby("Customer ID")["Total"].sum().reset_index().rename(columns={"Total": "Monetary"}),
    on="Customer ID", how="left"
)

# Average order value
tf["Avg_expenses_per_invoice"] = np.round(tf["Monetary"] / tf["Total Invoice"], 2)

# Customer lifespan
customer_lifespan = (
    df.groupby("Customer ID")["InvoiceDate"]
    .apply(lambda x: (x.max() - x.min()).days)
    .reset_index()
    .rename(columns={"InvoiceDate": "customer_span"})
)
tf = tf.merge(customer_lifespan, on="Customer ID", how="left")

# Drop intermediate InvoiceDate column
tf.drop(columns=["InvoiceDate"], inplace=True)

print(f"Customer-level table shape: {tf.shape}")
tf.head()


In [ ]:
# ── Average Purchase Gap (gap_days) ───────────────────────────────────────────
# Step 1: Invoice-level deduplicated timeline per customer
af = (
    df[["Invoice", "Customer ID", "InvoiceDate"]]
    .groupby(["Invoice", "Customer ID"])["InvoiceDate"]
    .min()
    .reset_index()
    .sort_values(["Customer ID", "InvoiceDate"])
)

# Step 2: Calculate time to next purchase
af["Next_Date"] = af.groupby("Customer ID")["InvoiceDate"].shift(-1)
af["gap_days"] = (af["Next_Date"] - af["InvoiceDate"]).dt.total_seconds() / (24 * 3600)

# Step 3: Average gap per customer
avg_gap = af.groupby("Customer ID")["gap_days"].mean().reset_index()

tf = tf.merge(avg_gap, on="Customer ID", how="left")

print("✅ gap_days engineered successfully.")
print(f"Customers with NaN gap_days (single-purchase): {tf['gap_days'].isna().sum():,}")
tf[["Customer ID", "Recency", "Total Invoice", "Monetary",
    "Avg_expenses_per_invoice", "customer_span", "gap_days"]].describe().round(2)


---

## 4. Churn Identification

### 🧠 Methodology

Rather than using a fixed recency cutoff (e.g., "inactive for 90 days = churned"), we build a **personalised, behaviour-driven churn flag** that accounts for each customer's natural buying rhythm.

---

### Why Not a Simple Threshold?

A blanket rule (e.g., "no purchase in 90 days") fails because:
- A customer who buys every 60 days and hasn't purchased in 80 days is **at risk**.
- A customer who buys quarterly and hasn't purchased in 80 days is **perfectly normal**.

---

### Churn Logic — Repeat Customers

For customers with multiple purchases, we compute a **dynamic gap threshold** that blends individual behaviour with the population trend:

$$
\text{Dynamic Threshold} = 0.85 \times \text{Customer Avg Gap} + 0.15 \times \text{Global Median Gap}
$$

A customer is flagged as **potential churn** if:

$$
\text{Recency} > 2 \times \text{Dynamic Threshold}
$$

The 2× multiplier adds a buffer, reducing false positives for customers who are only slightly past their normal cycle.

---

### Churn Logic — Single-Purchase Customers

Customers with only one purchase have no personal gap history. We use a **population benchmark**:

$$
\text{Churn if Recency} > 2.5 \times P_{85}(\text{gap\_days})
$$

where $P_{85}$ is the 85th percentile of all observed purchase gaps — a conservative threshold that avoids flagging recently acquired one-time buyers.


In [ ]:
# ── Churn Parameters ──────────────────────────────────────────────────────────
threshold   = tf["gap_days"].quantile(0.85)
global_gap  = tf["gap_days"].median()

print(f"85th percentile gap (threshold) : {threshold:.1f} days")
print(f"Global median gap               : {global_gap:.1f} days")


In [ ]:
# ── Flag: Single-Visit Churners ───────────────────────────────────────────────
tf["Potential_Churn_By_single_visit"] = np.where(
    (tf["Recency"] > 2.5 * threshold) & (tf["Total Invoice"] == 1),
    1, 0
)

n_single_churn = tf["Potential_Churn_By_single_visit"].sum()
print(f"Single-visit potential churners : {n_single_churn:,}")


In [ ]:
# ── Flag: Repeat Customer Churners ────────────────────────────────────────────
tf["Potential_Churn_over_all"] = np.where(
    tf["Recency"] > 0.85 * (2 * tf["gap_days"]) + 0.15 * global_gap,
    1, 0
)

n_repeat_churn = tf["Potential_Churn_over_all"].sum()
print(f"Repeat-customer potential churners : {n_repeat_churn:,}")


In [ ]:
# ── Combine into Final Churn Flag ─────────────────────────────────────────────
# OR logic: churned if either single-visit OR repeat condition is met
tf["Potential_Churn"] = np.where(
    (tf["Potential_Churn_By_single_visit"] == 1) | (tf["Potential_Churn_over_all"] == 1),
    1, 0
)

# Clean up intermediate columns
tf.drop(columns=["Potential_Churn_By_single_visit", "Potential_Churn_over_all"], inplace=True)

churn_rate = tf["Potential_Churn"].mean() * 100
active_count = (tf["Potential_Churn"] == 0).sum()
churn_count  = (tf["Potential_Churn"] == 1).sum()
total_count  = len(tf)

print("=" * 45)
print(f"  Total Customers      : {total_count:,}")
print(f"  Active Customers     : {active_count:,}  ({100 - churn_rate:.1f}%)")
print(f"  Potential Churners   : {churn_count:,}  ({churn_rate:.1f}%)")
print("=" * 45)


In [ ]:
# ── Visualise Churn Distribution ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Pie chart
labels = ["Active", "Potential Churn"]
sizes  = [active_count, churn_count]
colors = ["#4CAF50", "#F44336"]
axes[0].pie(sizes, labels=labels, colors=colors, autopct="%1.1f%%",
            startangle=90, wedgeprops={"edgecolor": "white", "linewidth": 2})
axes[0].set_title("Customer Churn Split", fontweight="bold")

# Bar chart — Recency distribution by churn status
sns.histplot(
    data=tf, x="Recency", hue="Potential_Churn",
    bins=40, kde=True, palette={0: "#4CAF50", 1: "#F44336"},
    ax=axes[1], alpha=0.65
)
axes[1].set_title("Recency Distribution by Churn Status", fontweight="bold")
axes[1].set_xlabel("Days Since Last Purchase (Recency)")
axes[1].legend(title="Churn", labels=["Active (0)", "Churned (1)"])

plt.suptitle("Behaviour-Driven Churn Identification", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()


### 📊 Churn Analysis — Key Findings

| Metric | Value |
|--------|-------|
| Total Customers | 4,365 |
| Single-purchase customers | 1,253 (28.7%) |
| Potential Churners | 1,149 |
| Estimated Churn Rate | **26.3%** |

**Interpretation:**  
- Nearly 1 in 4 customers shows churn-like behaviour.  
- The behaviour-based approach correctly avoids flagging recently acquired single-buyers.  
- Recency alone is insufficient — customers with naturally long buying cycles are not penalised unfairly.


---

## 5. Exploratory Data Analysis (EDA)

We analyse the **customer-level feature table** `tf` to understand distributions, outliers, and correlations before clustering.


In [ ]:
# ── Summary Statistics ────────────────────────────────────────────────────────
print("Customer-Level Feature Table — Summary Statistics")
tf.describe().round(2)


In [ ]:
# ── Univariate Distribution Plots ─────────────────────────────────────────────
features = {
    "Recency"                 : "Days Since Last Purchase",
    "Total Invoice"           : "Number of Orders (Frequency)",
    "Monetary"                : "Total Customer Spend (GBP)",
    "Avg_expenses_per_invoice": "Avg Spend per Order (GBP)",
    "customer_span"           : "Customer Lifespan (Days)",
    "gap_days"                : "Avg Days Between Purchases",
}

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for ax, (col, label) in zip(axes, features.items()):
    data = tf[col].dropna()
    # clip extreme outliers for display clarity only
    upper = data.quantile(0.99)
    sns.histplot(data[data <= upper], bins=35, kde=True, ax=ax,
                 color="#5B8DB8", edgecolor="white", linewidth=0.4)
    ax.set_title(label, fontweight="bold")
    ax.set_xlabel("")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

fig.suptitle("Univariate Distribution — Customer Features", fontsize=14,
             fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# ── Outlier Box Plots ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for ax, (col, label) in zip(axes, features.items()):
    sns.boxplot(y=tf[col].dropna(), ax=ax, color="#F0A500",
                flierprops=dict(marker="o", markerfacecolor="#E03030",
                                markersize=3, alpha=0.5))
    ax.set_title(label, fontweight="bold")
    ax.set_ylabel("")

fig.suptitle("Outlier Analysis — Customer Features", fontsize=14,
             fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()


### 📌 Outlier Decision

Extreme values were **retained intentionally** because:
- They represent genuine customer behaviour (e.g., enterprise buyers, high-frequency resellers).
- Removing them would destroy valuable business signal.
- K-Means with `StandardScaler` naturally handles scale differences.


In [ ]:
# ── Correlation Heatmap ───────────────────────────────────────────────────────
corr = tf.select_dtypes(exclude="object").corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))   # show only lower triangle
sns.heatmap(
    corr, mask=mask, annot=True, fmt=".2f",
    cmap="coolwarm", center=0, linewidths=0.5,
    annot_kws={"size": 9}, ax=ax,
    cbar_kws={"shrink": 0.75}
)
ax.set_title("Feature Correlation Matrix", fontsize=13, fontweight="bold", pad=12)
plt.tight_layout()
plt.show()


### 📊 EDA Key Insights

| Finding | Correlation | Business Implication |
|---------|-------------|----------------------|
| **Frequency → Revenue** | +0.63 | More orders = more spend; focus on repeat purchase conversion |
| **Recency → Churn** | +0.61 | Inactivity is the strongest churn signal |
| **Lifespan → Recency** (inverse) | −0.50 | Long-tenured customers stay more active |
| **Lifespan → Frequency** | +0.46 | Older customers accumulate more purchases naturally |
| **Gap Days → Churn** (inverse) | −0.36 | Personalised thresholds protect slow-cycle buyers from false positives |
| **Avg Order Value → Revenue** | weak | Revenue growth driven by *frequency*, not order size |


---

## 6. Customer Segmentation via K-Means

### 🛠️ Preprocessing for Clustering

**Challenge:** `gap_days` is `NaN` for single-purchase customers (no repeat purchase → no gap to measure).  
**Solution:** Preserve the information via a binary indicator, then impute with a high benchmark.


In [ ]:
# ── Prepare Clustering Dataset ────────────────────────────────────────────────
cluster_df = tf.drop(columns=["Customer ID", "Potential_Churn"]).copy()

# Binary flag: was this customer a single-purchase buyer?
cluster_df["single_purchase_customer"] = cluster_df["gap_days"].isna().astype(int)

# Impute NaN gap_days with 95th percentile (conservative "unknown/inactive" estimate)
p95_gap = cluster_df["gap_days"].quantile(0.95)
cluster_df["gap_days"] = cluster_df["gap_days"].fillna(p95_gap)

print(f"95th percentile gap used for imputation : {p95_gap:.1f} days")
print(f"Clustering features                     : {cluster_df.columns.tolist()}")
print(f"Any remaining nulls                     : {cluster_df.isnull().sum().sum()}")


In [ ]:
# ── Standardise Features ──────────────────────────────────────────────────────
scaler     = StandardScaler()
df_cluster = scaler.fit_transform(cluster_df)

print("✅ Features scaled with StandardScaler (mean=0, std=1).")


In [ ]:
# ── Elbow Method ──────────────────────────────────────────────────────────────
wcss = []
k_range = range(1, 16)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(df_cluster)
    wcss.append(km.inertia_)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(k_range, wcss, marker="o", linewidth=2, color="#5B8DB8",
        markerfacecolor="#E03030", markersize=8)
ax.axvline(x=8, linestyle="--", color="#E03030", alpha=0.7, label="Selected K=8")
ax.set_title("Elbow Method — Within-Cluster Sum of Squares (WCSS)", fontweight="bold")
ax.set_xlabel("Number of Clusters (K)")
ax.set_ylabel("WCSS (Inertia)")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# ── Silhouette Analysis ───────────────────────────────────────────────────────
sil_scores = []
k_range_sil = range(2, 12)

for k in k_range_sil:
    km     = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(df_cluster)
    score  = silhouette_score(df_cluster, labels)
    sil_scores.append(score)
    print(f"  K={k:2d}  →  Silhouette Score: {score:.4f}")

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(list(k_range_sil), sil_scores, marker="s", linewidth=2,
        color="#5B8DB8", markerfacecolor="#F0A500", markersize=9)
ax.axvline(x=8, linestyle="--", color="#E03030", alpha=0.7, label="Selected K=8")
ax.set_title("Silhouette Score vs Number of Clusters", fontweight="bold")
ax.set_xlabel("Number of Clusters (K)")
ax.set_ylabel("Silhouette Score")
ax.legend()
plt.tight_layout()
plt.show()


### 📐 Cluster Selection Rationale — K = 8

| Criterion | Evidence |
|-----------|----------|
| **Elbow Method** | WCSS improvement slows noticeably after K≈8 |
| **Silhouette Score** | Score > 0.40 at K=8, indicating well-separated clusters |
| **Business Interpretability** | 8 clusters produce distinct, actionable segments |
| **Balance** | Avoids under-segmentation (K=2) and over-fragmentation (K=12+) |


In [ ]:
# ── Fit Final K-Means Model ───────────────────────────────────────────────────
OPTIMAL_K = 8

kmeans = KMeans(n_clusters=OPTIMAL_K, random_state=42, n_init=10)
tf["Cluster"] = kmeans.fit_predict(df_cluster)

print(f"✅ K-Means fitted with K={OPTIMAL_K}")
print("\nCluster sizes:")
tf["Cluster"].value_counts().sort_index().to_frame("Customer Count")


In [ ]:
# ── Raw Cluster Profile Table ─────────────────────────────────────────────────
cluster_profile = tf.groupby("Cluster").agg(
    Customers       = ("Customer ID",            "count"),
    Recency_mean    = ("Recency",                "mean"),
    Frequency_mean  = ("Total Invoice",          "mean"),
    Monetary_mean   = ("Monetary",               "mean"),
    AvgOrder_mean   = ("Avg_expenses_per_invoice","mean"),
    Span_mean       = ("customer_span",          "mean"),
    Gap_mean        = ("gap_days",               "mean"),
    Churn_rate      = ("Potential_Churn",        "mean"),
).round(2)

print("Raw Cluster Profiles (before labelling):")
cluster_profile


---

## 7. Segment Profiles & Business Insights

Based on the raw cluster statistics above, each cluster is assigned a **descriptive business label**.


In [ ]:
# ── Assign Segment Labels ─────────────────────────────────────────────────────
segment_map = {
    0: "High-Value Repeat Customers",
    1: "Recent One-Time Buyers",
    2: "At-Risk Customers",
    3: "Enterprise Customers",
    4: "Data Outlier",
    5: "Lost One-Time Buyers",
    6: "Regular Active Customers",
    7: "VIP Customers",
}

tf["Segment"] = tf["Cluster"].map(segment_map)
print("✅ Segment labels assigned.")
tf["Segment"].value_counts()


In [ ]:
# ── Segment Summary Table ─────────────────────────────────────────────────────
segment_summary = tf.groupby("Segment").agg(
    Customers       = ("Customer ID",            "count"),
    Recency_avg     = ("Recency",                "mean"),
    Frequency_avg   = ("Total Invoice",          "mean"),
    Monetary_avg    = ("Monetary",               "mean"),
    AvgOrder_avg    = ("Avg_expenses_per_invoice","mean"),
    Lifespan_avg    = ("customer_span",          "mean"),
    Gap_avg         = ("gap_days",               "mean"),
    Churn_Rate      = ("Potential_Churn",        "mean"),
).round(2).reset_index()

print("Segment Summary:")
segment_summary.set_index("Segment")


In [ ]:
# ── Revenue Contribution by Segment ───────────────────────────────────────────
segment_revenue = (
    tf.groupby("Segment")["Monetary"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)
segment_revenue["Revenue_Share_%"] = (
    segment_revenue["Monetary"] / segment_revenue["Monetary"].sum() * 100
).round(1)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Revenue bar
palette = sns.color_palette("Set2", n_colors=len(segment_revenue))
bars = axes[0].barh(segment_revenue["Segment"], segment_revenue["Monetary"],
                    color=palette, edgecolor="white")
axes[0].set_xlabel("Total Revenue (GBP)", fontweight="bold")
axes[0].set_title("Revenue Contribution by Segment", fontweight="bold")
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"£{x/1e3:.0f}k"))
for bar, share in zip(bars, segment_revenue["Revenue_Share_%"]):
    axes[0].text(bar.get_width() * 1.01, bar.get_y() + bar.get_height() / 2,
                 f"{share}%", va="center", fontsize=9)

# Churn rate bar
churn_data = segment_summary.sort_values("Churn_Rate", ascending=False)
colors_churn = ["#E03030" if c > 0.5 else "#F0A500" if c > 0.2 else "#4CAF50"
                for c in churn_data["Churn_Rate"]]
axes[1].barh(churn_data["Segment"], churn_data["Churn_Rate"] * 100,
             color=colors_churn, edgecolor="white")
axes[1].set_xlabel("Potential Churn Rate (%)", fontweight="bold")
axes[1].set_title("Churn Rate by Segment", fontweight="bold")
axes[1].axvline(x=26.3, linestyle="--", color="grey", alpha=0.7,
                label="Overall avg (26.3%)")
axes[1].legend(fontsize=8)

plt.suptitle("Segment-Level Business Intelligence", fontsize=14,
             fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# ── Radar / Spider Chart — Normalised Segment Profiles ────────────────────────
from matplotlib.patches import FancyArrowPatch
import matplotlib.patches as mpatches

radar_features = ["Recency_avg", "Frequency_avg", "Monetary_avg",
                  "Lifespan_avg", "Gap_avg", "Churn_Rate"]
radar_labels   = ["Recency", "Frequency", "Monetary", "Lifespan", "Gap Days", "Churn Rate"]

# Normalise to 0-1 for each feature
radar_df = segment_summary.copy().set_index("Segment")
for col in radar_features:
    col_min, col_max = radar_df[col].min(), radar_df[col].max()
    radar_df[col] = (radar_df[col] - col_min) / (col_max - col_min + 1e-9)

N = len(radar_features)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]   # close the polygon

fig, axes = plt.subplots(2, 4, figsize=(18, 9),
                          subplot_kw=dict(polar=True))
axes = axes.flatten()
palette = sns.color_palette("Set2", n_colors=len(segment_summary))

for idx, (seg, ax, color) in enumerate(
    zip(segment_summary["Segment"], axes, palette)
):
    values = radar_df.loc[seg, radar_features].tolist()
    values += values[:1]

    ax.fill(angles, values, color=color, alpha=0.35)
    ax.plot(angles, values, color=color, linewidth=2)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(radar_labels, size=8)
    ax.set_yticks([0.25, 0.5, 0.75, 1.0])
    ax.set_yticklabels(["25%", "50%", "75%", "100%"], size=6)
    ax.set_title(seg, size=9, fontweight="bold", pad=12)

fig.suptitle("Segment Radar Profiles (Normalised Features)",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# ── Churn Count per Segment ───────────────────────────────────────────────────
churn_summary = (
    tf[tf["Potential_Churn"] == 1]
    .groupby("Segment")
    .size()
    .reset_index(name="Churned")
)
total_per_seg = tf.groupby("Segment").size().reset_index(name="Total")
churn_summary = churn_summary.merge(total_per_seg, on="Segment")
churn_summary["Churn_Rate_%"] = (churn_summary["Churned"] / churn_summary["Total"] * 100).round(1)
churn_summary.sort_values("Churn_Rate_%", ascending=False)


---

### 🏷️ Segment Profiles

<details>
<summary><b>🔴 At-Risk Customers</b> — Click to expand</summary>

- **Behaviour:** Moderate frequency, moderate spend, but recency is high relative to personal gap history.  
- **Risk Level:** 🔴 High churn concentration among repeat buyers.  
- **Action:** Win-back campaigns, personalised discount codes, re-engagement email flows.

</details>

<details>
<summary><b>🔴 Lost One-Time Buyers</b> — Click to expand</summary>

- **Behaviour:** Single purchase, long inactivity, outside 2.5× P85 threshold.  
- **Risk Level:** 🔴 Likely permanently lost.  
- **Action:** Aggressive win-back, or reallocate budget to retain active customers.

</details>

<details>
<summary><b>🟡 Recent One-Time Buyers</b> — Click to expand</summary>

- **Behaviour:** Single purchase, low recency (recently acquired).  
- **Risk Level:** 🟡 Uncertain — not yet churned but no repeat signal.  
- **Action:** First-to-second purchase nudges, personalised recommendations.

</details>

<details>
<summary><b>🟢 High-Value Repeat Customers</b> — Click to expand</summary>

- **Behaviour:** High frequency, high spend, long lifespan.  
- **Risk Level:** 🟢 Low churn.  
- **Action:** Loyalty programmes, early access to new products.

</details>

<details>
<summary><b>🟢 Regular Active Customers</b> — Click to expand</summary>

- **Behaviour:** Moderate-to-good frequency, consistent spend, long lifespan.  
- **Risk Level:** 🟢 Very low churn.  
- **Action:** Cross-sell, upsell, loyalty rewards.

</details>

<details>
<summary><b>💎 VIP Customers</b> — Click to expand</summary>

- **Behaviour:** Extremely high spend, very high frequency, recent activity.  
- **Risk Level:** 🟢 Minimal churn.  
- **Action:** VIP membership, exclusive offers, premium customer experience.

</details>

<details>
<summary><b>🏢 Enterprise Customers</b> — Click to expand</summary>

- **Behaviour:** Highest order frequency, very high spend.  
- **Risk Level:** 🟢 Very low churn.  
- **Action:** Dedicated account management, custom bulk pricing, SLA agreements.

</details>

<details>
<summary><b>⚠️ Data Outlier</b> — Click to expand</summary>

- **Behaviour:** Negative monetary values (net returns exceed purchases).  
- **Risk Level:** N/A — likely erroneous or exceptional accounting records.  
- **Action:** Flag for data quality review; exclude from marketing campaigns.

</details>


---

## 8. PCA Visualisation

Principal Component Analysis (PCA) reduces the 7-dimensional feature space to 2D for interpretable visualisation of cluster separation.


In [ ]:
# ── PCA Projection ────────────────────────────────────────────────────────────
pca = PCA(n_components=2, random_state=42)
pca_data = pca.fit_transform(df_cluster)

pca_df = pd.DataFrame(pca_data, columns=["PC1", "PC2"])
pca_df["Segment"] = tf["Segment"].values

variance_explained = pca.explained_variance_ratio_.sum() * 100
print(f"Variance explained by PC1 + PC2 : {variance_explained:.2f}%")
print(f"  PC1 : {pca.explained_variance_ratio_[0]*100:.2f}%")
print(f"  PC2 : {pca.explained_variance_ratio_[1]*100:.2f}%")


In [ ]:
# ── PCA Scatter Plot ──────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 8))

palette = sns.color_palette("Set2", n_colors=pca_df["Segment"].nunique())

sns.scatterplot(
    data=pca_df, x="PC1", y="PC2", hue="Segment",
    palette=palette, alpha=0.65, s=40, linewidth=0, ax=ax
)

ax.set_title(
    f"Customer Segments in PCA Space  (Variance Explained: {variance_explained:.1f}%)",
    fontsize=13, fontweight="bold"
)
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)", fontweight="bold")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)", fontweight="bold")
ax.legend(title="Segment", bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=9)

plt.tight_layout()
plt.show()


In [ ]:
# ── PCA Feature Loading Arrows (Biplot) ───────────────────────────────────────
components = pca.components_
feature_names = cluster_df.columns.tolist()

fig, ax = plt.subplots(figsize=(10, 8))

sns.scatterplot(
    data=pca_df, x="PC1", y="PC2", hue="Segment",
    palette=palette, alpha=0.35, s=20, linewidth=0, ax=ax, legend=False
)

scale = 4   # arrow scale factor
for i, feat in enumerate(feature_names):
    ax.annotate(
        "", xy=(components[0, i] * scale, components[1, i] * scale),
        xytext=(0, 0),
        arrowprops=dict(arrowstyle="->", color="#333333", lw=1.5)
    )
    ax.text(components[0, i] * scale * 1.1,
            components[1, i] * scale * 1.1,
            feat, fontsize=8, color="#333333", ha="center")

ax.axhline(0, color="grey", lw=0.5, linestyle="--")
ax.axvline(0, color="grey", lw=0.5, linestyle="--")
ax.set_title("PCA Biplot — Feature Loadings on PC1 & PC2", fontweight="bold")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
plt.tight_layout()
plt.show()


---

## 9. Key Takeaways & Recommendations

### 📊 Summary Statistics

| Metric | Value |
|--------|-------|
| Dataset size (raw) | ~1M transactions |
| Customers analysed | 4,365 |
| Estimated churn rate | **26.3%** |
| Clusters identified | **8** |
| PCA variance retained | **~61%** |

---

### 🎯 Priority Actions by Segment

| Priority | Segment | Action |
|----------|---------|--------|
| 🔴 Urgent | At-Risk Customers | Win-back campaigns within 30 days |
| 🔴 High | Lost One-Time Buyers | Final win-back attempt; re-evaluate spend |
| 🟡 Medium | Recent One-Time Buyers | Nurture with first repeat purchase incentive |
| 🟢 Retain | VIP + Enterprise | Exclusive loyalty perks, dedicated support |
| 🟢 Grow | Regular + High-Value | Cross-sell, upsell, referral programmes |

---

### 💡 Analytical Contributions

1. **Behaviour-driven churn flag** — Blends individual purchase rhythm with population benchmark, outperforming a simple recency threshold.
2. **`gap_days` feature** — Novel purchase cycle feature that captures buying rhythm and distinguishes casual from habitual customers.
3. **Principled missing value handling** — `single_purchase_customer` indicator preserves signal before imputation.
4. **K selection via dual criteria** — Elbow + Silhouette balance statistical quality and business interpretability.

---

### 🔭 Next Steps

- [ ] Deploy churn scores to a CRM for automated re-engagement triggers.
- [ ] A/B test win-back campaign effectiveness across At-Risk vs Lost segments.
- [ ] Incorporate product category data for deeper behavioural profiling.
- [ ] Retrain segmentation model quarterly as new transactions arrive.
- [ ] Build a customer lifetime value (CLV) prediction model on top of these features.

---

<div align="center">

*Analysis by — Customer Segmentation & Churn Modelling Project*  
*Dataset: Online Retail II (UCI ML Repository)*  
*Tools: Python · Pandas · Scikit-learn · Seaborn · Matplotlib*

</div>
